# 01 - Predicting GO Membrane Properties from XPS (Section 3.2)

**Physical question:** Can the C1s XPS fingerprint of a GO membrane predict its structural and
surface properties - in particular wet d-spacing (which controls nanofiltration selectivity)?

**Model:** Ridge regression (α = 10), features = C-C, C-O, C=O bond fractions.  
For wet d-spacing, dry d-spacing is added as a 4th feature (physically motivated: dry structure
is a known predictor of swelling).

**Cross-validation hierarchy:**
- **Random** (50 × 5-fold KFold) - optimistic: same-GO-batch membranes in both train and test
- **GroupKFold** (50 shuffles, groups = doi × material_label) - isolates same GO synthesis batch
- **LOAO** (leave-one-article-out) - strictest: model never sees any data from held-out article

Data split by GO **source**: commercial GO (Sigma, XFNano, …) vs self-synthesised GO.
Their XPS fingerprints occupy different regions of feature space - mixing them conflates two
different structure–chemistry relationships.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
%matplotlib inline

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

DATA_FILE = os.path.join('..', 'go_filtration_data.csv')

# Config
ALPHA       = 10
N_SEEDS     = 50
N_GKF_SEEDS = 50
K_FOLDS     = 5
MIN_TRAIN   = 5
MIN_N       = 10
MIN_DOIS    = 3

XPS      = ['C-C', 'C-O', 'C=O']                       # C:O_from_C1s omitted: collinear
XPS_PLUS = ['C-C', 'C-O', 'C=O', 'dry_dspacing_nm']   # for wet d-spacing model

# Targets: (column, features, display_label, unit)
TARGETS = [
    ('wet_dspacing_nm',     XPS_PLUS, 'wet d-spacing (XPS + dry d-sp.)', 'nm'),
    ('wet_dspacing_nm',     XPS,      'wet d-spacing (XPS only)',         'nm'),
    ('Water_contact_angle', XPS,      'Water contact angle',              '°'),
    ('Zeta_potential_mV',   XPS,      'Zeta potential',                   ' mV'),
    ('dry_dspacing_nm',     XPS,      'dry d-spacing',                    'nm'),
]

SOURCES    = ['Commercial',  'Self-synth']
SRC_LABELS = ['Commercial GO', 'Self-synthesised GO']
COL_COMM   = '#2166AC'
COL_SELF   = '#D6604D'

print('Config loaded.')

In [ ]:
# --- Load data ---
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig')
for c in ['C:O_from_C1s', 'C-C', 'C-O', 'C=O',
          'wet_dspacing_nm', 'dry_dspacing_nm',
          'Zeta_potential_mV', 'Water_contact_angle']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['Source'] = df['supplier'].str.lower().str.startswith('self-synth').map(
    {True: 'Self-synth', False: 'Commercial'})

# One row per unique membrane
df_mem = df.drop_duplicates(subset=['doi', 'membrane_uid']).copy().reset_index(drop=True)
print(f'Dataset: {len(df_mem)} unique membranes, {df_mem["doi"].nunique()} articles')
print(f'  Commercial: {(df_mem["Source"]=="Commercial").sum()}')
print(f'  Self-synth: {(df_mem["Source"]=="Self-synth").sum()}')

In [ ]:
# --- CV functions ---
def make_model():
    return make_pipeline(StandardScaler(), Ridge(alpha=ALPHA))

def make_cb():
    return CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                             verbose=False, random_seed=42)

def prepare(sub, feats, col):
    sub = sub.copy().dropna(subset=feats + [col])
    sub = sub.drop_duplicates(subset=['doi'] + feats + [col])
    return sub.reset_index(drop=True)

def cv_random(sub, feats, col):
    r2s = []
    X, y_arr = sub[feats].values, sub[col].values
    for seed in range(N_SEEDS):
        preds = np.full(len(y_arr), np.nan)
        for tr_idx, te_idx in KFold(n_splits=K_FOLDS, shuffle=True,
                                    random_state=seed).split(X):
            if len(tr_idx) < MIN_TRAIN: continue
            m = make_model()
            m.fit(X[tr_idx], y_arr[tr_idx])
            preds[te_idx] = m.predict(X[te_idx])
        v = ~np.isnan(preds)
        r2s.append(r2_score(y_arr[v], preds[v]))
    return float(np.mean(r2s))

def cv_groupkfold(sub, feats, col):
    pairs = list(sub.groupby(['doi', 'material_label']).groups.keys())
    r2s = []
    for seed in range(N_GKF_SEEDS):
        rng = np.random.default_rng(seed)
        shuffled = pairs.copy(); rng.shuffle(shuffled)
        p2f  = {p: i % K_FOLDS for i, p in enumerate(shuffled)}
        fold = sub.apply(lambda r: p2f[(r['doi'], r['material_label'])], axis=1).values
        preds = np.full(len(sub), np.nan)
        for f in range(K_FOLDS):
            tm = fold != f; vm = fold == f
            if tm.sum() < MIN_TRAIN: continue
            m = make_model()
            m.fit(sub.iloc[tm][feats], sub.iloc[tm][col])
            preds[vm] = m.predict(sub.iloc[vm][feats])
        v = ~np.isnan(preds)
        r2s.append(r2_score(sub[col].values[v], preds[v]))
    return float(np.mean(r2s))

def cv_loao(sub, feats, col, model_fn=None):
    if model_fn is None:
        model_fn = make_model
    preds = np.full(len(sub), np.nan)
    for doi_out in sub['doi'].unique():
        tm = sub['doi'] != doi_out
        vm = sub['doi'] == doi_out
        if tm.sum() < MIN_TRAIN: continue
        m = model_fn()
        m.fit(sub.loc[tm, feats], sub.loc[tm, col])
        preds[vm] = m.predict(sub.loc[vm, feats])
    v    = ~np.isnan(preds)
    y    = sub[col].values
    r2   = r2_score(y[v], preds[v])
    rmse = float(np.sqrt(mean_squared_error(y[v], preds[v])))
    mae  = float(mean_absolute_error(y[v], preds[v]))
    return r2, rmse, mae, preds

print('CV functions defined.')

## Ridge vs CatBoost - LOAO comparison (Table S6)

Comparison on the three XPS-predictable targets - **wet d-spacing, WCA, Zeta potential** -
using XPS features only (C–C, C–O, C=O), separately for Commercial and Self-synthesised GO.
Dry d-spacing is intentionally excluded here to isolate the contribution of XPS chemistry alone.

Ridge regression outperformed CatBoost in 4 of 6 combinations. With only 3 features and
~15-30 membranes per source, CatBoost has insufficient data to learn non-linear interactions -
Ridge benefits from its implicit regularisation via the linear assumption.

In [ ]:
# --- Ridge vs CatBoost LOAO: wet d-spacing (XPS only), WCA, Zeta × 2 sources ---
COMPARE_TARGETS = [
    ('wet_dspacing_nm',     XPS, 'wet d-spacing', 'nm'),
    ('Water_contact_angle', XPS, 'WCA',            '°'),
    ('Zeta_potential_mV',   XPS, 'Zeta potential', 'mV'),
]

rows = []
for col, feats, label, unit in COMPARE_TARGETS:
    for src in SOURCES:
        sub   = prepare(df_mem[df_mem['Source'] == src], feats, col)
        n     = len(sub)
        n_doi = sub['doi'].nunique()
        if n < MIN_N or n_doi < MIN_DOIS:
            rows.append(dict(Target=label, Source=src, n=n, skip=True))
            continue

        r2_ridge, rmse_ridge, _, _ = cv_loao(sub, feats, col, model_fn=make_model)
        r2_cb,    rmse_cb,    _, _ = cv_loao(sub, feats, col, model_fn=make_cb)

        rows.append(dict(
            Target=label, Source=src, n=n,
            **{f'Ridge R²': round(r2_ridge, 3)},
            **{f'CB R²':    round(r2_cb,    3)},
            **{f'RMSE Ridge ({unit})': round(rmse_ridge, 3)},
            **{f'RMSE CB ({unit})':    round(rmse_cb,    3)},
            **{'Δ R² (R−CB)': round(r2_ridge - r2_cb, 3)},
            **{'Better': 'Ridge' if r2_ridge >= r2_cb else 'CatBoost'},
            skip=False,
        ))

comp_df = pd.DataFrame([r for r in rows if not r.get('skip')])

print('=== LOAO R²: Ridge vs CatBoost (XPS only, 3 features) ===\n')
display_cols = ['Target', 'Source', 'n', 'Ridge R²', 'CB R²', 'Δ R² (R−CB)', 'Better']
print(comp_df[display_cols].to_string(index=False))

ridge_wins = (comp_df['Better'] == 'Ridge').sum()
print(f'\nRidge wins: {ridge_wins} / {len(comp_df)}')

In [ ]:
# --- Run all target × source combinations ---
# This takes ~1-2 min (50 × 50 CV runs per combination)
records    = []
loao_store = {}   # (label, src) -> (sub_prepared, col, loao_preds, rmse)

print(f'{"Target":<32} {"Source":<12} {"n":>4} {"dois":>4} '
      f'{"Random":>8} {"GroupKF":>8} {"LOAO":>8}  RMSE')
print('-' * 90)

for col, feats, label, unit in TARGETS:
    for src in SOURCES:
        sub   = prepare(df_mem[df_mem['Source'] == src], feats, col)
        n     = len(sub)
        n_doi = sub['doi'].nunique()

        if n < MIN_N or n_doi < MIN_DOIS:
            print(f'  {label:<30} {src:<12} {n:>4}  SKIP (n too small)')
            continue

        r2_rand              = cv_random(sub, feats, col)
        r2_gkf               = cv_groupkfold(sub, feats, col)
        r2_loao, rmse, mae, loao_preds = cv_loao(sub, feats, col)

        records.append(dict(
            target=label, source=src, n=n, n_doi=n_doi,
            r2_rand=r2_rand, r2_gkf=r2_gkf, r2_loao=r2_loao,
            rmse_loao=rmse, mae_loao=mae,
        ))
        loao_store[(label, src)] = (sub, col, loao_preds, rmse)

        print(f'  {label:<30} {src:<12} {n:>4} {n_doi:>4} '
              f'  {r2_rand:>+7.3f}  {r2_gkf:>+7.3f}  {r2_loao:>+7.3f}'
              f'  {rmse:.3f}{unit.strip()}')

results_df = pd.DataFrame(records)

## Figure - Parity plot: wet d-spacing from XPS + dry d-spacing (LOAO)

Each colour = one article. Grey band = ±RMSE.  
Self-synthesised GO (LOAO R² ≈ +0.44): XPS captures enough variance to predict wet swelling across articles.  
Commercial GO (LOAO R² ≈ −0.17): narrow XPS range → model cannot distinguish between articles.

In [ ]:
KEY = 'wet d-spacing (XPS + dry d-sp.)'

fig_b, axes_b = plt.subplots(1, 2, figsize=(10, 4.8))
fig_b.subplots_adjust(wspace=0.32)

for ax, src, src_label in zip(axes_b, SOURCES, SRC_LABELS):
    key = (KEY, src)
    if key not in loao_store:
        ax.text(0.5, 0.5, 'insufficient data', transform=ax.transAxes,
                ha='center', va='center', color='0.5', fontsize=10)
        ax.set_title(src_label); continue

    sub, col, preds, band = loao_store[key]
    v      = ~np.isnan(preds)
    actual = sub[col].values[v]
    pred   = preds[v]
    sub_v  = sub[v].copy()

    r2   = r2_score(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    dois = sub_v['doi'].unique()
    ndoi = len(dois)
    cmap_art = plt.cm.get_cmap('tab20', max(ndoi, 1))
    dcol = {d: cmap_art(i) for i, d in enumerate(dois)}

    for doi_val in dois:
        mask = sub_v['doi'].values == doi_val
        ax.scatter(actual[mask], pred[mask], color=dcol[doi_val], s=52,
                   edgecolors='white', linewidths=0.4, alpha=0.92, zorder=5)

    lo = min(actual.min(), pred.min()) - 0.05
    hi = max(actual.max(), pred.max()) + 0.05
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.1, alpha=0.4, zorder=3)
    ax.fill_between([lo, hi], [lo-band, hi-band], [lo+band, hi+band],
                    color='k', alpha=0.06, zorder=2)

    sign = '+' if r2 >= 0 else ''
    ax.text(0.05, 0.96, f'$R^2 = {sign}{r2:.2f}$  (LOAO-CV)',
            transform=ax.transAxes, fontsize=10.5, va='top', fontweight='bold')
    ax.text(0.05, 0.88, f'RMSE = {rmse:.3f} nm   MAE = {mae:.3f} nm',
            transform=ax.transAxes, fontsize=8.5, va='top', color='0.4')
    ax.text(0.05, 0.81, f'n = {v.sum()} membranes  ·  {ndoi} articles',
            transform=ax.transAxes, fontsize=8.5, va='top', color='0.4')
    ax.text(0.05, 0.74, 'features: C–C, C–O, C=O, dry d-sp.',
            transform=ax.transAxes, fontsize=7.8, va='top', color='0.5')

    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
    ax.set_xlabel('Actual wet d-spacing (nm)', fontsize=10)
    ax.set_ylabel('Predicted wet d-spacing (nm)', fontsize=10)
    ax.set_title(src_label, fontsize=11, pad=7)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=9)

fig_b.text(0.5, -0.02,
           'Each colour = one article (DOI)  ·  grey band = ±RMSE',
           ha='center', fontsize=8.5, color='0.5')
plt.show()

## SI Figure - Parity grid: all targets × both GO sources (LOAO)

In [ ]:
fig_si, axes_si = plt.subplots(len(TARGETS), 2, figsize=(10, 5.0 * len(TARGETS)))
fig_si.subplots_adjust(hspace=0.50, wspace=0.32)

for row, (col, feats, label, unit) in enumerate(TARGETS):
    for c, (src, src_label) in enumerate(zip(SOURCES, SRC_LABELS)):
        ax = axes_si[row, c]
        key = (label, src)

        if key not in loao_store:
            ax.text(0.5, 0.5, 'insufficient data', transform=ax.transAxes,
                    ha='center', va='center', color='0.5', fontsize=10)
            ax.set_title(src_label, fontsize=10.5)
            continue

        sub, col_name, preds, band = loao_store[key]
        v      = ~np.isnan(preds)
        actual = sub[col_name].values[v]
        pred   = preds[v]
        sub_v  = sub[v].copy()

        r2   = r2_score(actual, pred)
        rmse = np.sqrt(mean_squared_error(actual, pred))
        ndoi = sub_v['doi'].nunique()

        dois = sub_v['doi'].unique()
        cmap_art = plt.cm.get_cmap('tab20', max(len(dois), 1))
        dcol = {d: cmap_art(i) for i, d in enumerate(dois)}

        for doi_val in dois:
            mask = sub_v['doi'].values == doi_val
            ax.scatter(actual[mask], pred[mask], color=dcol[doi_val], s=38,
                       edgecolors='white', linewidths=0.35, alpha=0.92, zorder=5)

        lo = min(actual.min(), pred.min()); hi = max(actual.max(), pred.max())
        pad = (hi - lo) * 0.07; lo -= pad; hi += pad
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1.0, alpha=0.4)
        ax.fill_between([lo, hi], [lo-band, hi-band], [lo+band, hi+band],
                        color='k', alpha=0.06)

        r2_color = '#1a7a1a' if r2 > 0 else '#cc2222'
        sign = '+' if r2 >= 0 else ''
        ax.text(0.05, 0.97, f'$R^2 = {sign}{r2:.2f}$',
                transform=ax.transAxes, fontsize=10, va='top',
                fontweight='bold', color=r2_color)
        ax.text(0.05, 0.88, f'RMSE = {rmse:.3f}{unit}',
                transform=ax.transAxes, fontsize=8, va='top', color='0.45')
        ax.text(0.05, 0.81, f'n = {v.sum()}  ·  {ndoi} art',
                transform=ax.transAxes, fontsize=8, va='top', color='0.45')

        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
        ax.set_xlabel(f'Actual ({unit.strip()})', fontsize=9.5)
        ax.set_ylabel(f'Predicted ({unit.strip()})', fontsize=9.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=9)

        title = f'{src_label}\n{label}' if c == 0 else src_label
        ax.set_title(title, fontsize=10.5, pad=7,
                     fontweight='bold' if c == 0 else 'normal')

fig_si.text(0.5, -0.005,
            'LOAO-CV  ·  Ridge regression (α = 10)  ·  each colour = one article',
            ha='center', fontsize=9, color='0.5')
plt.show()

In [ ]:
# --- Summary table ---
print(results_df[['target','source','n','n_doi',
                   'r2_rand','r2_gkf','r2_loao','rmse_loao']]
      .round(3).to_string(index=False))